In [1]:
from chunking_evaluation import BaseChunker
import os
import re
from prediction_evaluation.chunkers import (
    SentenceChunker,
    CharacterChunker,
    TokenChunker,
    RecursiveCharacterChunker,
    ResTokenChunker,
    KamradtChunker,
    ClusterChunker,
    LLMChunker
)
from llama_index.core import VectorStoreIndex
from llama_index.core.schema import Document
from langchain.embeddings import HuggingFaceEmbeddings
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings
from typing import Dict,List
import sys,os
sys.path.append(os.path.abspath("../../"))
from backend.database.core.funcs import get_documents_by_theme,get_document_themes
from backend.database.config.config import settings
new_path = os.getcwd().split('\\')[0:-1]
new_path = '\\'.join(p for p in new_path)
print(new_path)


c:\Users\johnk\Documents\GitHub\AILA-application\backend


In [14]:
files = {}
themes = get_document_themes()
for theme in themes:
    files[theme] = []
    theme_files = get_documents_by_theme(theme=theme)['documents']
    for file in theme_files:
        files[theme].append(file)

# for file in files['Phishing Scenarios']:
#     print(file)

# print(files.keys())
for key in files.keys():
    print(key,len(files[key]))

[<backend.database.entities.document_theme.Document_Theme object at 0x000002A3300CAB10>, <backend.database.entities.document_theme.Document_Theme object at 0x000002A3300CBE90>, <backend.database.entities.document_theme.Document_Theme object at 0x000002A3300C9F10>, <backend.database.entities.document_theme.Document_Theme object at 0x000002A3300CA630>]
General Data Protection Regulation 272
Greek Cybercrime Legislation 8
Law Cases 15
Phishing Scenarios 7


### Splitting documents

In [22]:
def parse_phishing(files:Dict[str,str],chunker:BaseChunker):
    chunks = []
    counter = 0
    for file in files:
        text = file['content']
        title = file['title']
        attack_type = title.split('.txt')[0]
        text_chunks = chunker.split_text(text)
        for chunk in text_chunks:
            chunks.append(
                {
                    "id": f"phishing_{counter}",
                    "content": chunk,
                    "metadata": {
                        "source": "Phishing Scenarios",
                        "doc_type": "explainer",
                        "title": attack_type,
                        "lang": "en",
                    }
                }
            )
            counter+=1
    return chunks

def parse_gdpr(files:Dict[str,str],chunker:BaseChunker):
    chunks = []
    counter = 0
    for file in files:
        text = file['content']
        title = file['title']
        title = title.split('.txt')[0]
        text_chunks = chunker.split_text(text)
        for chunk in text_chunks:
            chunks.append(
                {
                    "id": f"gdpr_{counter}",
                    "content": chunk,
                    "metadata": {
                        "source": "GDPR",
                        "doc_type": "regulation",
                        "title": title,
                        "lang": "en"
                    }
                }
            )   
            counter+=1

    return chunks

def parse_law_cases(files:Dict[str,str],chunker:BaseChunker):
    chunks = []
    counter = 0
    case_id_ = 0
    for file in files:
        
        case = file['content']        
        match = re.search(r"Decision number:\s*(.*?)\n", case)
        case_id = match.group(1).strip() if match else f"case_{case_id_}"
        court = re.search(r"Court \(Civil/Criminal\):\s*(.*?)\n", case)
        court_type = court.group(1).strip().lower() if court else "unknown"
        outcome = re.search(r"Outcome \(innocent, guilty\):\s*(.*?)\n", case)
        laws = re.findall(r"Law\s+\d+/\d+|Article\s+\d+[A-Z]?(\s+of\s+Law\s+\d+/\d+)?", case)
        title = file['title']
        title = title.split('.txt')[0]
        text_chunks = chunker.split_text(case)
        for chunk in text_chunks:
            chunks.append({
                "id": f"case_{counter}",
                "content": chunk,
                "metadata": {
                    "title":title,
                    "source": "Greek Court Decisions",
                    "doc_type": "case_law",
                    "jurisdiction": "GR",
                    "case_id": case_id,
                    "civil_or_criminal": court_type,
                    "outcome": outcome.group(1).strip() if outcome else "unknown",
                    "relevant_laws": list(set(laws)),
                    "lang": "en"
                }
            })
            counter+=1
        case_id_ +=1 

    return chunks

def parse_cybercrime(files:Dict[str,str],chunker:BaseChunker):
    chunks = []
    counter = 0
    for file in files:
        title = file['title']
        title = title.split("//")[-1].split('.txt')[0]
        article_id = re.findall(r"Article\s+(\d+[A-Z]?)", title)
        law_id = re.findall(r"[ΝΠΚ]\.?\s?\d+/?\d*", title)
        text = file['content']
        text_chunks = chunker.split_text(text)
        for chunk in text_chunks:
            chunks.append({
                "id": f"cybercrime_{counter}",
                "content": chunk,
                "metadata": {
                    "title":title,
                    "source": "Greek Cybercrime Law",
                    "doc_type": "criminal_statute",
                    "law": law_id[0] if law_id else "unknown",
                    "article_number": article_id[0] if article_id else str(counter),
                    "lang": "en",
                    "jurisdiction": "GR"
                }
            })
            counter+=1

    return chunks


In [23]:
phishing_chunks = parse_phishing(files['Phishing Scenarios'],SentenceChunker(sentences_per_chunk=1))

law_cases_chunks_recall = parse_law_cases(files['Law Cases'],TokenChunker(tokens_per_chunk=1000,overlap=100))
law_cases_chunks_precision = parse_law_cases(files['Law Cases'],RecursiveCharacterChunker(characters_per_chunk=100,overlap=0))
law_cases_chunks_precision_openai = parse_law_cases(files['Law Cases'],ResTokenChunker(tokens_per_chunk=100,overlap=0))
law_cases_chunks_best_iou = parse_law_cases(files['Law Cases'],RecursiveCharacterChunker(characters_per_chunk=400,overlap=0))

gpc_chunks_recall = parse_cybercrime(files['Greek Cybercrime Legislation'],SentenceChunker(sentences_per_chunk=20))
gpc_chunks_precision = parse_cybercrime(files['Greek Cybercrime Legislation'],ResTokenChunker(tokens_per_chunk=200,overlap=20))
gpc_chunks = parse_cybercrime(files['Greek Cybercrime Legislation'],RecursiveCharacterChunker(characters_per_chunk=400,overlap=200))

gdpr_chunks_recall = parse_gdpr(files['General Data Protection Regulation'],SentenceChunker(sentences_per_chunk=20))
gdpr_chunks_precision = parse_gdpr(files['General Data Protection Regulation'],ResTokenChunker(tokens_per_chunk=200,overlap=20))
gdpr_chunks = parse_cybercrime(files['General Data Protection Regulation'],SentenceChunker(sentences_per_chunk=1))

### Index Creation

In [24]:
Settings.llm = None
### Phishing Scenarios
### Chunks
phishing_documents = []
for chunk in phishing_chunks:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    phishing_documents.append(Document(text=text,metadata=metadata))

### Index
phishing_index = VectorStoreIndex.from_documents(
    documents=phishing_documents,
    embed_model=OpenAIEmbedding(model="text-embedding-3-large")  # or "text-embedding-3-large"
)
phishing_index.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/phishing_index_documents_openai')


phishing_index = VectorStoreIndex.from_documents(
    documents = phishing_documents,
    embed_model = HuggingFaceEmbeddings(model_name = 'IoannisKat1/multilingual-e5-large-ft-new',model_kwargs = {"trust_remote_code":True})
)
phishing_index.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/phishing_index_documents_trained_embedding')

LLM is explicitly disabled. Using MockLLM.


You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


In [25]:
### Law Cases
law_cases_recall_documents = []
for chunk in law_cases_chunks_recall:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    law_cases_recall_documents.append(Document(text=text,metadata=metadata))

law_cases_index_recall = VectorStoreIndex.from_documents(
    documents = law_cases_recall_documents,
    embed_model = HuggingFaceEmbeddings(model_name='IoannisKat1/modernbert-embed-base-ft-new',model_kwargs = {"trust_remote_code":True})
)
law_cases_index_recall.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/law_cases_recall_index_documents_trained_embedding')


law_cases_precision_documents = []
for chunk in law_cases_chunks_precision:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    law_cases_precision_documents.append(Document(text=text,metadata=metadata))

law_cases_index_precision = VectorStoreIndex.from_documents(
    documents = law_cases_precision_documents,
    embed_model = HuggingFaceEmbeddings(model_name='IoannisKat1/bge-m3-ft-new',model_kwargs = {"trust_remote_code":True})
)
law_cases_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/law_cases_precision_index_documents_trained_embedding')


law_cases_precision_documents = []
for chunk in law_cases_chunks_precision_openai:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    law_cases_precision_documents.append(Document(text=text,metadata=metadata))

law_cases_index_precision = VectorStoreIndex.from_documents(
    documents = law_cases_precision_documents,
    embed_model = OpenAIEmbedding(model="text-embedding-3-large")
)
law_cases_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/law_cases_precision_openai_index_documents')


law_cases_documents = []
for chunk in law_cases_chunks_best_iou:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    law_cases_documents.append(Document(text=text,metadata=metadata))

law_cases_index_precision = VectorStoreIndex.from_documents(
    documents = law_cases_documents,
    embed_model = HuggingFaceEmbeddings(model_name='IoannisKat1/all-mpnet-base-v2-ft-new',model_kwargs = {"trust_remote_code":True})
)
law_cases_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/law_cases_best_iou_index_documents')



You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


In [26]:
# ### Greek Penal Code
gpc_recall_documents = []
for chunk in gpc_chunks_recall:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    gpc_recall_documents.append(Document(text=text,metadata=metadata))

gpc_index_recall = VectorStoreIndex.from_documents(
    documents = gpc_recall_documents,
    embed_model = HuggingFaceEmbeddings(model_name="IoannisKat1/all-MiniLM-L6-v2-ft-new",model_kwargs = {"trust_remote_code":True})
)
gpc_index_recall.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/gpc_recall_index_documents_trained_embedding')



gpc_precision_documents = []
for chunk in gpc_chunks_precision:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    gpc_precision_documents.append(Document(text=text,metadata=metadata))

gpc_index_precision = VectorStoreIndex.from_documents(
    documents = gpc_precision_documents,
    embed_model = HuggingFaceEmbeddings(model_name='IoannisKat1/multilingual-e5-large-ft-new',model_kwargs = {"trust_remote_code":True})
)
gpc_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/gpc_precision_index_documents_trained_embedding')


gpc_documents = []
for chunk in gpc_chunks:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    gpc_documents.append(Document(text=text,metadata=metadata))

gpc_index_precision = VectorStoreIndex.from_documents(
    documents = gpc_documents,
    embed_model = HuggingFaceEmbeddings(model_name='IoannisKat1/all-mpnet-base-v2-ft-new',model_kwargs = {"trust_remote_code":True})
)
gpc_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/gpc_best_iou_index_documents')


You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.


In [27]:
# ### GDPR
gdpr_recall_documents = []
for chunk in gdpr_chunks_recall:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    gdpr_recall_documents.append(Document(text=text,metadata=metadata))

gdpr_index_recall = VectorStoreIndex.from_documents(
    documents = gdpr_recall_documents,
    embed_model = HuggingFaceEmbeddings(model_name='IoannisKat1/modernbert-embed-base-ft-new',model_kwargs = {"trust_remote_code":True})
)
gdpr_index_recall.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/gdpr_recall_index_documents_trained_embedding')


gdpr_precision_documents = []
for chunk in gdpr_chunks_precision:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    gdpr_precision_documents.append(Document(text=text,metadata=metadata))

gdpr_index_precision = VectorStoreIndex.from_documents(
    documents = gdpr_precision_documents,
    embed_model = HuggingFaceEmbeddings(model_name = 'IoannisKat1/multilingual-e5-large-ft-new',model_kwargs = {"trust_remote_code":True})
)
gdpr_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/gdpr_precision_index_documents_trained_embedding')


gdpr_documents = []
for chunk in gdpr_chunks:
    metadata = {
        'id':chunk['id']
    }
    for key in chunk['metadata'].keys():
        metadata[key] = chunk['metadata'][key]
    text = chunk['content']
    gdpr_documents.append(Document(text=text,metadata=metadata))

gdpr_index_precision = VectorStoreIndex.from_documents(
    documents = gdpr_documents,
    embed_model = HuggingFaceEmbeddings(model_name = 'IoannisKat1/modernbert-embed-base-ft-new',model_kwargs = {"trust_remote_code":True})
)
gdpr_index_precision.storage_context.persist(persist_dir=f'{new_path}/vector_indexes_new/gdpr_best_iou_index_documents')



You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
You are trying to use a model that was created with Sentence Transformers version 5.1.2, but you're currently using version 4.1.0. This might cause unexpected behavior or errors. In that case, try to update to the latest version.
